<div dir="rtl">

# منصة تحويل الكتب العربية إلى Markdown

شغّل الخلايا الثلاث بالترتيب (زر ▶)، وستحصل على **رابط للمنصة** تفتحه من هاتفك.

1. **التثبيت** — عدة دقائق، مرة واحدة لكل جلسة.
2. **ربط Drive** (اختياري لكن مُستحسن) — لحفظ النتائج في Drive.
3. **تشغيل المنصة** — يطبع رابطًا اضغطه: ترفع PDF، تتابع التقدم، تقرأ الكتاب، تنزّل الناتج.

⚠ **مهم:**
- اترك تبويب Colab مفتوحًا أثناء العمل — إغلاقه يوقف المنصة.
- الرابط يتغيّر في كل تشغيل جديد.
- المعالجة على خوادم Google لا على جهازك.

</div>

In [ ]:
#@title ١) التثبيت { display-mode: "form" }
%pip install -q "paddlepaddle==3.3.1" "paddleocr[doc-parser]==3.7.0" pymupdf fastapi uvicorn python-multipart markdown
!rm -rf /content/book_ocr && git clone -q https://github.com/7aidaraa/book_ocr /content/book_ocr
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("\u2713 التثبيت اكتمل — شغّل الخلية التالية")

In [ ]:
#@title ٢) ربط Google Drive لحفظ النتائج (اختياري) { display-mode: "form" }
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# نجعل مجلد النتائج داخل المنصة يشير إلى Drive مباشرة
target = Path("/content/drive/MyDrive/كتب-محوّلة")
target.mkdir(parents=True, exist_ok=True)

output_link = Path("/content/book_ocr/data/output")
output_link.parent.mkdir(parents=True, exist_ok=True)
if output_link.is_symlink() or output_link.exists():
    if output_link.is_symlink():
        output_link.unlink()
    else:
        import shutil; shutil.rmtree(output_link)
os.symlink(target, output_link)

print(f"\u2713 النتائج ستُحفظ في Drive داخل: كتب-محوّلة/")

In [ ]:
#@title ٣) تشغيل المنصة والحصول على الرابط { display-mode: "form" }
import os, re, subprocess, sys, time, urllib.request

os.chdir("/content/book_ocr")

server = subprocess.Popen(
    [sys.executable, "run.py"],
    env={**os.environ, "HOST": "0.0.0.0", "PORT": "8000"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print("تشغيل الخادم...")
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/", timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            raise SystemExit("\u2717 فشل تشغيل الخادم:\n" + server.stdout.read())
        time.sleep(1)
else:
    raise SystemExit("\u2717 الخادم لم يستجب")

print("فتح الرابط العام...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:
        break
    match = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise SystemExit("\u2717 تعذر إنشاء الرابط — أعد تشغيل هذه الخلية")

print("\n" + "=" * 52)
print("  \u2713 المنصة تعمل — افتح هذا الرابط:")
print(f"  {public_url}")
print("=" * 52)
print("\n\u26a0 اترك هذه الخلية تعمل ولا تغلق التبويب.")
print("   لإيقاف المنصة: اضغط زر التوقف \u25a0 في هذه الخلية.\n")
print("--- سجل الخادم ---")
for line in server.stdout:
    print(line, end="")